# 03c — Audio ↔ text matching на актуальном корпусе

Один прогон на прежней лучшей конфигурации без parameter sweep: word TF-IDF, группы по 60 секунд, окно 400 символов, шаг 100, порог 0.20. Matching выполняется по логическому потоку чтения: внешние примечания из FB2 разворачиваются у их внутренних ссылок.

> **Deprecated результаты:** `outputs/besy/run_01/matching_experiments/` и `outputs/besy/run_01/best_result_analysis/` были получены на прежнем неполном корпусе длиной 1 086 475 символов. После обновления `01_text_normalization.ipynb` их абсолютные позиции, метрики и выводы несовместимы с текущим корпусом. Не используйте их как anchors, оценку качества или точку сравнения для этого прогона.

Старые артефакты не изменяются. Этот ноутбук сохраняет новые результаты отдельно в `outputs/besy/run_01/current_corpus_best_result_analysis/`.

Метрики повторяют 03b: matched/unmatched, распределение score, нарушения монотонности, стыки аудиофайлов, покрытие текста и набор примеров для ручной проверки.

In [1]:
from __future__ import annotations

import json
import os
import pickle
from bisect import bisect_right
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path(os.environ.get(
    'SPARK_ROOT',
    '/home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit',
))
RUN_DIR = PROJECT_ROOT / 'outputs/besy/run_01'
OUTPUT_DIR = RUN_DIR / 'current_corpus_best_result_analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_PATH = RUN_DIR / 'normalized_text.pkl'
AUDIO_PATH = RUN_DIR / 'audio_segments.pkl'
assert TEXT_PATH.is_file(), f'Не найден текстовый корпус: {TEXT_PATH}'
assert AUDIO_PATH.is_file(), f'Не найдены аудиосегменты: {AUDIO_PATH}'

GROUP_SIZE = 60
WINDOW_SIZE = 400
WINDOW_STEP = 100
TOP_K = 3
THRESHOLD = 0.20
BATCH_SIZE = 128

print(f'Результаты: {OUTPUT_DIR}')


Результаты: /home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit/outputs/besy/run_01/current_corpus_best_result_analysis


In [2]:
with TEXT_PATH.open('rb') as file:
    text_data = pickle.load(file)
with AUDIO_PATH.open('rb') as file:
    audio_data = pickle.load(file)

normalized_text = text_data['normalized_text']
matching_text = text_data['matching_text']
matching_segments = text_data['matching_segments']
section_boundaries = text_data['section_boundaries']
audio_segments = audio_data['audio_segments']
total_duration = audio_data['total_duration']

assert matching_segments, 'Логический matching-поток пуст.'
print(f'Канонический текст: {len(normalized_text):,} символов; блоков: {text_data["paragraph_count"]:,}')
print(f'Логический matching-поток: {len(matching_text):,} символов; сегментов: {len(matching_segments):,}')
print(f'Аудио: {len(audio_segments):,} сегментов; длительность: {total_duration / 3600:.1f} ч')


Канонический текст: 1,881,785 символов; блоков: 8,123
Логический matching-поток: 1,881,557 символов; сегментов: 7,836
Аудио: 35,260 сегментов; длительность: 36.2 ч


In [3]:
def build_audio_groups(segments, group_size_sec=60, min_group_sec=10):
    """Собрать последовательные ASR-сегменты в группы, как в 03b."""
    groups = []
    buf_text, buf_start, buf_end, buf_file, buf_idx = [], None, None, None, None
    for index, segment in enumerate(segments):
        if buf_start is None:
            buf_start = segment['global_start']
            buf_file = segment['file']
            buf_idx = segment['file_index']
        buf_text.append(segment['text'])
        buf_end = segment['global_end']
        duration = buf_end - buf_start
        has_next = index + 1 < len(segments)
        file_boundary = not has_next or segments[index + 1]['file_index'] != buf_idx
        if file_boundary and duration < min_group_sec:
            continue
        if duration >= group_size_sec or file_boundary:
            groups.append({
                'audio_start': buf_start, 'audio_end': buf_end, 'duration': duration,
                'text': ' '.join(buf_text), 'file': buf_file, 'file_index': buf_idx,
                'segment_count': len(buf_text),
            })
            buf_text, buf_start = [], None
    if buf_text:
        groups.append({
            'audio_start': buf_start, 'audio_end': buf_end, 'duration': buf_end - buf_start,
            'text': ' '.join(buf_text), 'file': buf_file, 'file_index': buf_idx,
            'segment_count': len(buf_text),
        })
    return groups


def build_text_windows(text, window_size=400, step=100):
    if len(text) <= window_size:
        return [{'char_start': 0, 'char_end': len(text), 'text': text}]
    windows = []
    for start in range(0, len(text) - window_size + 1, step):
        windows.append({'char_start': start, 'char_end': start + window_size, 'text': text[start:start + window_size]})
    if windows[-1]['char_end'] < len(text):
        windows.append({'char_start': len(text) - window_size, 'char_end': len(text), 'text': text[-window_size:]})
    return windows


matching_segment_starts = [segment['match_char_start'] for segment in matching_segments]


def resolve_matching_position(match_char_pos):
    """Перевести координату matching-потока в логическую позицию чтения."""
    index = bisect_right(matching_segment_starts, match_char_pos) - 1
    index = max(0, min(index, len(matching_segments) - 1))
    segment = matching_segments[index]
    if segment['kind'] == 'note':
        reading_position = segment['reading_position']
    else:
        offset = max(0, match_char_pos - segment['match_char_start'])
        reading_position = segment['reading_position'] + offset
    return {
        'reading_position': reading_position,
        'section': segment['source'],
        'kind': segment['kind'],
        'target_id': segment.get('target_id'),
    }


def json_safe(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {key: json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    return value


In [4]:
audio_groups = build_audio_groups(audio_segments, group_size_sec=GROUP_SIZE)
text_windows = build_text_windows(matching_text, WINDOW_SIZE, WINDOW_STEP)
group_texts = [group['text'] for group in audio_groups]
window_texts = [window['text'] for window in text_windows]

print(f'Групп: {len(audio_groups):,}; окон: {len(text_windows):,}')
print('Строю word TF-IDF…', end=' ', flush=True)
vectorizer = TfidfVectorizer(analyzer='word', max_features=50_000)
matrix = vectorizer.fit_transform(window_texts + group_texts)
window_matrix = matrix[:len(text_windows)]
group_matrix = matrix[len(text_windows):]
print('OK')

all_candidates = []
for start in range(0, group_matrix.shape[0], BATCH_SIZE):
    stop = min(start + BATCH_SIZE, group_matrix.shape[0])
    scores = cosine_similarity(group_matrix[start:stop], window_matrix)
    for row in scores:
        indexes = np.argpartition(row, -TOP_K)[-TOP_K:]
        indexes = indexes[np.argsort(row[indexes])[::-1]]
        candidates = []
        for index in indexes:
            window = text_windows[int(index)]
            location = resolve_matching_position(window['char_start'])
            candidates.append({
                'window_index': int(index), 'score': float(row[index]),
                'match_char_start': window['char_start'], 'match_char_end': window['char_end'],
                **location,
            })
        all_candidates.append(candidates)
    print(f'  {stop:,}/{group_matrix.shape[0]:,}', end='\r', flush=True)
print()

# Выбираем одну последовательность кандидатов: большой шаг назад хуже близкого по score варианта.
decoder_scores = [candidate['score'] for candidate in all_candidates[0]]
decoder_paths = [[index] for index in range(TOP_K)]
for group_index, candidates in enumerate(all_candidates[1:], start=1):
    next_scores, next_paths = [], []
    for candidate_index, candidate in enumerate(candidates):
        transitions = []
        for previous_index, previous_score in enumerate(decoder_scores):
            previous_candidate = all_candidates[group_index - 1][decoder_paths[previous_index][-1]]
            backward = max(0, previous_candidate['reading_position'] - candidate['reading_position'] - WINDOW_STEP)
            penalty = min(2.0, 0.05 + backward / 100_000) if backward else 0.0
            transitions.append((previous_score - penalty, previous_index))
        best_transition, previous_index = max(transitions, key=lambda item: item[0])
        next_scores.append(best_transition + candidate['score'])
        next_paths.append(decoder_paths[previous_index] + [candidate_index])
    decoder_scores, decoder_paths = next_scores, next_paths
selected_candidate_indexes = decoder_paths[int(np.argmax(decoder_scores))]
decoder_adjusted = sum(index != 0 for index in selected_candidate_indexes)

audio_map, unmatched = [], []
for group_index, group in enumerate(audio_groups):
    candidate = all_candidates[group_index][selected_candidate_indexes[group_index]]
    if candidate['score'] < THRESHOLD:
        unmatched.append({
            'group_index': group_index, 'audio_start': group['audio_start'], 'audio_end': group['audio_end'],
            'duration': group['duration'], 'file': group['file'], 'file_index': group['file_index'],
            'text': group['text'][:300], 'best_score': candidate['score'], 'candidates': all_candidates[group_index],
        })
        continue
    audio_map.append({
        'group_index': group_index, 'audio_start': group['audio_start'], 'audio_end': group['audio_end'],
        'char_start': candidate['reading_position'], 'char_end': candidate['reading_position'] + WINDOW_SIZE,
        'match_char_start': candidate['match_char_start'], 'match_char_end': candidate['match_char_end'],
        'file': group['file'], 'file_index': group['file_index'], 'score': candidate['score'],
        'section': candidate['section'], 'match_kind': candidate['kind'], 'target_id': candidate['target_id'],
        'audio_text': group['text'][:200],
        'matched_text': matching_text[candidate['match_char_start']:candidate['match_char_start'] + 200],
        'candidates': all_candidates[group_index],
    })

print(f'Matched: {len(audio_map):,}; unmatched: {len(unmatched):,} ({100 * len(unmatched) / len(audio_groups):.1f}%)')
print(f'Последовательный decoder выбрал не лучший локальный кандидат: {decoder_adjusted:,} раз')


Групп: 2,088; окон: 18,813
Строю word TF-IDF… OK
  2,088/2,088
Matched: 2,088; unmatched: 0 (0.0%)
Последовательный decoder выбрал не лучший локальный кандидат: 2 раз


In [5]:
violations = []
for previous, current in zip(audio_map, audio_map[1:]):
    if current['char_start'] < previous['char_start']:
        violations.append({
            'prev_group': previous['group_index'], 'curr_group': current['group_index'],
            'prev_time': previous['audio_start'], 'curr_time': current['audio_start'],
            'prev_pos': previous['char_start'], 'curr_pos': current['char_start'],
            'jump_back_chars': previous['char_start'] - current['char_start'],
            'prev_section': previous['section'], 'curr_section': current['section'],
            'prev_score': previous['score'], 'curr_score': current['score'],
        })

boundary_issues = []
for previous, current in zip(audio_map, audio_map[1:]):
    if current['file_index'] != previous['file_index']:
        boundary_issues.append({
            'transition': f"{previous['file']} → {current['file']}",
            'prev_char_end': previous['char_end'], 'curr_char_start': current['char_start'],
            'gap_chars': current['char_start'] - previous['char_end'],
            'prev_section': previous['section'], 'curr_section': current['section'],
        })

matched_scores = [item['score'] for item in audio_map]
unmatched_scores = [item['best_score'] for item in unmatched]
positions = [item['char_start'] for item in audio_map]
times = [item['audio_start'] for item in audio_map]
gaps = [item['gap_chars'] for item in boundary_issues]

summary = {
    'status': 'current_corpus',
    'parameters': {
        'group_size_sec': GROUP_SIZE, 'window_size': WINDOW_SIZE, 'window_step': WINDOW_STEP,
        'top_k': TOP_K, 'threshold': THRESHOLD, 'tfidf_method': 'word',
    },
    'corpus': {
        'text_characters': len(normalized_text), 'text_blocks': text_data['paragraph_count'],
        'section_boundaries': len(section_boundaries),
        'matching_characters': len(matching_text), 'matching_segments': len(matching_segments),
        'resolved_note_targets': len(text_data.get('note_targets', {})),
    },
    'audio': {'segments': len(audio_segments), 'duration_seconds': total_duration},
    'matching': {
        'groups': len(audio_groups), 'windows': len(text_windows), 'matched': len(audio_map),
        'unmatched': len(unmatched), 'unmatched_percent': 100 * len(unmatched) / len(audio_groups),
        'matched_score': {
            'min': min(matched_scores), 'mean': float(np.mean(matched_scores)),
            'median': float(np.median(matched_scores)), 'max': max(matched_scores),
        },
        'unmatched_score': ({
            'min': min(unmatched_scores), 'mean': float(np.mean(unmatched_scores)),
            'median': float(np.median(unmatched_scores)), 'max': max(unmatched_scores),
        } if unmatched_scores else None),
        'decoder_adjusted_candidates': decoder_adjusted,
    },
    'monotonicity': {
        'violations': len(violations),
        'violation_percent': 100 * len(violations) / max(1, len(audio_map) - 1),
    },
    'file_boundaries': {
        'count': len(boundary_issues),
        'gap_min': min(gaps) if gaps else None, 'gap_mean': float(np.mean(gaps)) if gaps else None,
        'gap_median': float(np.median(gaps)) if gaps else None, 'gap_max': max(gaps) if gaps else None,
    },
    'timestamp': datetime.now().isoformat(),
}

print(json.dumps(json_safe(summary), ensure_ascii=False, indent=2))

print('\nUnmatched по файлам:')
for filename, count in Counter(item['file'] for item in unmatched).most_common(15):
    print(f'  {count:>3} — {filename}')

print('\nКрупнейшие скачки назад:')
for item in sorted(violations, key=lambda value: value['jump_back_chars'], reverse=True)[:10]:
    print(f"  {item['jump_back_chars']:>9,} симв | {item['prev_section']} → {item['curr_section']}")

print('\nГлавы с наихудшим средним score:')
scores_by_section = {}
for item in audio_map:
    scores_by_section.setdefault(item['section'], []).append(item['score'])
for section, scores in sorted(scores_by_section.items(), key=lambda item: np.mean(item[1]))[:10]:
    print(f'  avg={np.mean(scores):.3f}; n={len(scores):>4} — {section}')


{
  "status": "current_corpus",
  "parameters": {
    "group_size_sec": 60,
    "window_size": 400,
    "window_step": 100,
    "top_k": 3,
    "threshold": 0.2,
    "tfidf_method": "word"
  },
  "corpus": {
    "text_characters": 1881785,
    "text_blocks": 8123,
    "section_boundaries": 999,
    "matching_characters": 1881557,
    "matching_segments": 7836,
    "resolved_note_targets": 965
  },
  "audio": {
    "segments": 35260,
    "duration_seconds": 130333.07428571432
  },
  "matching": {
    "groups": 2088,
    "windows": 18813,
    "matched": 2088,
    "unmatched": 0,
    "unmatched_percent": 0.0,
    "matched_score": {
      "min": 0.2818945081225308,
      "mean": 0.7516699246919251,
      "median": 0.7609509719327872,
      "max": 0.9554401067644515
    },
    "unmatched_score": null,
    "decoder_adjusted_candidates": 2
  },
  "monotonicity": {
    "violations": 0,
    "violation_percent": 0.0
  },
  "file_boundaries": {
    "count": 23,
    "gap_min": -122,
    "gap_mean"

In [6]:
(OUTPUT_DIR / 'summary.json').write_text(
    json.dumps(json_safe(summary), ensure_ascii=False, indent=2), encoding='utf-8'
)
with (OUTPUT_DIR / 'audio_map.pkl').open('wb') as file:
    pickle.dump(audio_map, file)
(OUTPUT_DIR / 'unmatched.json').write_text(
    json.dumps(json_safe(unmatched), ensure_ascii=False, indent=2), encoding='utf-8'
)
(OUTPUT_DIR / 'violations.json').write_text(
    json.dumps(json_safe(violations), ensure_ascii=False, indent=2), encoding='utf-8'
)
(OUTPUT_DIR / 'boundary_issues.json').write_text(
    json.dumps(json_safe(boundary_issues), ensure_ascii=False, indent=2), encoding='utf-8'
)

rng = np.random.RandomState(123)
check_indexes = sorted(rng.choice(len(audio_map), min(20, len(audio_map)), replace=False))
with (OUTPUT_DIR / 'manual_check.txt').open('w', encoding='utf-8') as file:
    file.write('=== СЛУЧАЙНЫЕ ТОЧКИ ДЛЯ РУЧНОЙ ПРОВЕРКИ ===\n\n')
    for index in check_indexes:
        item = audio_map[index]
        minute, second = divmod(int(item['audio_start']), 60)
        file.write(f'[{minute:3d}:{second:02d}] {item["file"]}\n')
        file.write(f'  Раздел: {item["section"]}\n')
        file.write(f'  Score: {item["score"]:.3f}\n')
        file.write(f'  Аудио: {item["audio_text"]}\n')
        file.write(f'  FB2: {item["matched_text"]}\n\n')
    file.write('=== КРУПНЕЙШИЕ НАРУШЕНИЯ МОНОТОННОСТИ ===\n\n')
    for item in sorted(violations, key=lambda value: value['jump_back_chars'], reverse=True)[:15]:
        file.write(f"Скачок назад: {item['jump_back_chars']:,} символов\n")
        file.write(f"  {item['prev_section']} → {item['curr_section']}\n\n")

print(f'Артефакты сохранены в {OUTPUT_DIR}:')
print('  summary.json, audio_map.pkl, unmatched.json, violations.json, boundary_issues.json, manual_check.txt')


Артефакты сохранены в /home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit/outputs/besy/run_01/current_corpus_best_result_analysis:
  summary.json, audio_map.pkl, unmatched.json, violations.json, boundary_issues.json, manual_check.txt
